In [1]:
import polars as pl

In [2]:
variant_ids_file = "/s/project/deeprvat/ukb_gym/variant_files/qced_maf1e-3_loftee_olink_genes_unique_variants.parquet"

vep_inputs_prefix = "/s/project/deeprvat/ukb_gym/variant_files/variant_metadata_chunked/variant_metadata_chunk_"
lines_per_chunk = 500_000

In [3]:
!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes_unique_variants.parquet -o {variant_ids_file}

Error: path "/s/project/deeprvat/ukb_gym/variant_files/qced_maf1e-3_loftee_oli
nk_genes_unique_variants.parquet" already exists but -f/--overwrite was not
set


In [4]:
# Get the total number of rows in the Parquet file
try:
    num_rows = pl.scan_parquet(variant_ids_file).select(pl.len()).collect().item()
    print(f"Estimated total number of rows: {num_rows}")
except Exception as e:
    print(f"Error getting row count: {e}")
    print("Please ensure the file is accessible and not corrupted.")
    exit()

num_chunks = (num_rows + lines_per_chunk - 1) // lines_per_chunk
print(f"Total number of chunks to process: {num_chunks}")

for i in range(num_chunks):
    offset = i * lines_per_chunk
    limit = lines_per_chunk

    print(
        f"Processing chunk {i + 1} (rows {offset + 1} to {min(offset + limit, num_rows)})"
    )

    try:
        chunk_lf = pl.scan_parquet(variant_ids_file).slice(offset, limit)
        chunk_df = chunk_lf.collect()

        if not chunk_df.is_empty():
            output_filename = f"{vep_inputs_prefix}{i + 1}.vcf"  # Use .vcf extension
            with open(output_filename, "w") as f:
                # Write the simplified VCF header for all chunks
                f.write("##fileformat=VCFv4.0\n")
                f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
                for row in chunk_df.iter_rows(named=True):
                    variant_id = row.get("ID", ".")

                    chrom = variant_id.split(":")[0]
                    pos = variant_id.split(":")[1]
                    ref = variant_id.split(":")[2]
                    alt = variant_id.split(":")[3]
                    vcf_row = f"{chrom}\t{pos}\t{variant_id}\t{ref}\t{alt}\t.\t.\t.\n"
                    f.write(vcf_row)
            print(f"Saved chunk {i + 1} to {output_filename}")
        else:
            print(f"Chunk {i + 1} is empty, skipping save.")

    except pl.ColumnNotFoundError as e:
        print(f"Error: Column not found in chunk {i + 1} - {e}")
        print("Please check the column names in your Parquet file.")
        break
    except Exception as e:
        print(f"Error processing chunk {i + 1}: {e}")
        break

print("Chunk-wise processing and saving to simplified VCF format complete.")

Estimated total number of rows: 148687476
Total number of chunks to process: 298
Processing chunk 1 (rows 1 to 500000)


Saved chunk 1 to /s/project/deeprvat/ukb_gym/variant_files/variant_metadata_chunked/variant_metadata_chunk_1.vcf
Processing chunk 2 (rows 500001 to 1000000)
Saved chunk 2 to /s/project/deeprvat/ukb_gym/variant_files/variant_metadata_chunked/variant_metadata_chunk_2.vcf
Processing chunk 3 (rows 1000001 to 1500000)
Saved chunk 3 to /s/project/deeprvat/ukb_gym/variant_files/variant_metadata_chunked/variant_metadata_chunk_3.vcf
Processing chunk 4 (rows 1500001 to 2000000)
Saved chunk 4 to /s/project/deeprvat/ukb_gym/variant_files/variant_metadata_chunked/variant_metadata_chunk_4.vcf
Processing chunk 5 (rows 2000001 to 2500000)
Saved chunk 5 to /s/project/deeprvat/ukb_gym/variant_files/variant_metadata_chunked/variant_metadata_chunk_5.vcf
Processing chunk 6 (rows 2500001 to 3000000)
Saved chunk 6 to /s/project/deeprvat/ukb_gym/variant_files/variant_metadata_chunked/variant_metadata_chunk_6.vcf
Processing chunk 7 (rows 3000001 to 3500000)
Saved chunk 7 to /s/project/deeprvat/ukb_gym/variant_